# MapReduce Matrix Multiplication

## Learning Objectives

By the end of this notebook you will be able to:

1. **Implement** one-pass matrix-matrix multiplication in MapReduce
2. **Implement** two-pass matrix multiplication for very large matrices
3. **Derive** the communication cost of each approach
4. **Choose** between one-pass and two-pass based on matrix dimensions and memory constraints

## Problem Setup

Compute $P = M \times N$ where:
- $M$ is an $m \times k$ matrix
- $N$ is a $k \times n$ matrix  
- $P$ is an $m \times n$ matrix

$$P_{ij} = \sum_{l=1}^{k} M_{il} \cdot N_{lj}$$

Both $M$ and $N$ are stored in the distributed filesystem as sparse triples $(i, j, v)$.
The result $P$ may be dense even if inputs are sparse.

## One-Pass Algorithm

Compute the entire product in a single MapReduce job.

**Map:**
- For each entry $M_{il} = v$: emit key $= (i, j)$ for **all** $j \in [n]$, value $= (M, l, v)$
- For each entry $N_{lj} = v$: emit key $= (i, j)$ for **all** $i \in [m]$, value $= (N, l, v)$

Wait — that's $O(mn)$ pairs per non-zero entry. Instead, use the natural join approach:

**Better Map:**
- For $M_{il} = v$: emit key $= l$, value $= (M, i, v)$
- For $N_{lj} = v$: emit key $= l$, value $= (N, j, v)$

**Reduce:** for each $l$, pair every $M$ value $(M, i, v)$ with every $N$ value $(N, j, v')$,
emitting $(i, j, v \cdot v')$ pairs. Then sum within $(i, j)$... but that requires a second pass.

**True one-pass:** emit key $= (i, j)$ directly:
- For $M_{il}$: emit $(i, j), (M, l, v)$ for each $j$  
- For $N_{lj}$: emit $(i, j), (N, l, v)$ for each $i$

Reducer for key $(i,j)$: match on $l$ and sum products.

In [ ]:
import numpy as np
from collections import defaultdict


def matmul_one_pass(M_sparse, N_sparse, m, k, n):
    """
    One-pass MapReduce matrix multiplication P = M @ N.

    M_sparse : list of (i, l, value)
    N_sparse : list of (l, j, value)
    m, k, n  : dimensions (M is m×k, N is k×n)
    """
    # Map phase
    # For M[i,l]=v → emit (i,j) for all j: key=(i,j), val=(M, l, v)
    # For N[l,j]=v → emit (i,j) for all i: key=(i,j), val=(N, l, v)
    intermediate = defaultdict(lambda: {"M": {}, "N": {}})

    for i, l, v in M_sparse:
        for j in range(n):
            intermediate[(i, j)]["M"][l] = v

    for l, j, v in N_sparse:
        for i in range(m):
            intermediate[(i, j)]["N"][l] = v

    # Reduce phase: for each (i,j), sum M[i,l]*N[l,j] over shared l
    P = np.zeros((m, n))
    for (i, j), sides in intermediate.items():
        for l in sides["M"]:
            if l in sides["N"]:
                P[i, j] += sides["M"][l] * sides["N"][l]

    return P


# Verify
M = np.array([[1, 2, 0],
              [0, 3, 4]], dtype=float)
N = np.array([[5, 0],
              [1, 2],
              [0, 3]], dtype=float)

M_sparse = [(i, l, M[i,l]) for i in range(M.shape[0]) for l in range(M.shape[1]) if M[i,l] != 0]
N_sparse = [(l, j, N[l,j]) for l in range(N.shape[0]) for j in range(N.shape[1]) if N[l,j] != 0]

P_mr = matmul_one_pass(M_sparse, N_sparse, 2, 3, 2)
P_np = M @ N
print("One-pass result:\n", P_mr)
print("NumPy result:\n",    P_np)
print("Match:", np.allclose(P_mr, P_np))

## Cost of One-Pass

For dense $M$ ($m \times k$) and $N$ ($k \times n$):

**Communication cost:**

$$Q_{\text{one-pass}} = m \cdot k \cdot n + k \cdot n \cdot m = 2mkn$$

Each $M_{il}$ entry is replicated $n$ times (once per column of $N$).
Each $N_{lj}$ entry is replicated $m$ times (once per row of $M$).

**Reducer size:** each key $(i,j)$ receives $2k$ values (one $M$ and one $N$ value per $l$).

$$q_{\text{one-pass}} = 2k \text{ values per reducer}$$

This is manageable when $k$ is small, but $Q = 2mkn$ is large for large matrices.

## Two-Pass Algorithm

Reduce communication cost by splitting the computation into two MapReduce jobs.

**Pass 1** — join $M$ and $N$ on the shared index $l$:
- Map: $M_{il} \to$ key $= l$, value $= (M, i, v)$; $N_{lj} \to$ key $= l$, value $= (N, j, v)$  
- Reduce: for each $l$, emit all products $(i, j, M_{il} \cdot N_{lj})$

**Pass 2** — sum products for each $(i,j)$:
- Map: identity (pass through $(i,j,\text{product})$)
- Reduce: for each $(i,j)$, sum all products $\to P_{ij}$

**Why two passes?** Pass 1 produces partial products grouped by $l$.
Pass 2 groups partial products by $(i,j)$ and sums them.

In [ ]:
def matmul_two_pass(M_sparse, N_sparse, m, k, n):
    """
    Two-pass MapReduce matrix multiplication P = M @ N.
    Pass 1: join on shared index l → partial products
    Pass 2: group by (i,j) → sum partial products
    """
    # ── Pass 1 ──────────────────────────────────────────────────────────────
    # Map
    by_l = defaultdict(lambda: {"M": [], "N": []})
    for i, l, v in M_sparse:
        by_l[l]["M"].append((i, v))
    for l, j, v in N_sparse:
        by_l[l]["N"].append((j, v))

    # Reduce: emit partial products (i, j, M_il * N_lj)
    partial_products = []
    for l, sides in by_l.items():
        for i, m_val in sides["M"]:
            for j, n_val in sides["N"]:
                partial_products.append((i, j, m_val * n_val))

    # ── Pass 2 ──────────────────────────────────────────────────────────────
    # Map (identity)
    by_ij = defaultdict(list)
    for i, j, prod in partial_products:
        by_ij[(i, j)].append(prod)

    # Reduce: sum
    P = np.zeros((m, n))
    for (i, j), prods in by_ij.items():
        P[i, j] = sum(prods)

    return P


P_mr2 = matmul_two_pass(M_sparse, N_sparse, 2, 3, 2)
print("Two-pass result:\n", P_mr2)
print("NumPy result:\n",    P_np)
print("Match:", np.allclose(P_mr2, P_np))

## Cost of Two-Pass

**Pass 1 communication cost:**

$$Q_1 = mk + kn$$

Each entry of $M$ emitted once; each entry of $N$ emitted once. This is just the input size.

**Pass 1 reducer size:**

$$q_1 = \text{(M entries for l)} \times \text{(N entries for l)} \approx m \cdot n \text{ per } l$$

Pass 1 reducer must hold all rows of $M$ and all columns of $N$ for one value of $l$: 
$q_1 \approx m + n$ entries per reducer.

**Pass 2 communication cost:**

$$Q_2 = mkn \quad \text{(all partial products from Pass 1)}$$

**Pass 2 reducer size:**

$$q_2 = k \quad \text{(k partial products per (i,j) pair)}$$

**Total:** $Q_{\text{two-pass}} = mk + kn + mkn \approx mkn$

Asymptotically the same as one-pass, but with much smaller $q$ in Pass 1.

## Comparison

| Property | One-Pass | Two-Pass |
|----------|----------|---------|
| Number of MR jobs | 1 | 2 |
| Communication cost $Q$ | $2mkn$ | $mk + kn + mkn \approx mkn$ |
| Reducer size $q$ | $2k$ values per $(i,j)$ | Pass 1: $m+n$; Pass 2: $k$ |
| Memory requirement | Low (only $k$ values per reducer) | Pass 1 may need $m+n$ values |
| Best when | $k$ is small, matrix is sparse | $k$ is large, $m,n$ moderate |

**Rule of thumb:**
- Use **one-pass** when $k$ (the shared dimension) is small enough that each Reducer
  can hold $k$ values in memory
- Use **two-pass** when $k$ is large but $m$ and $n$ (per-$l$ slice) fit in memory

In [ ]:
def compare_costs(m, k, n, density=1.0):
    """
    Compare communication costs for one-pass vs two-pass.
    density: fraction of non-zero entries (1.0 = dense)
    """
    nnz_M = m * k * density
    nnz_N = k * n * density

    Q_one  = 2 * nnz_M * n + 2 * nnz_N * m    # oversimplified for sparse
    Q_one  = 2 * m * k * n * density

    Q_two  = nnz_M + nnz_N + m * k * n * density ** 2

    print(f"  m={m}, k={k}, n={n}, density={density}")
    print(f"  Q one-pass  = {Q_one:>15,.0f}")
    print(f"  Q two-pass  = {Q_two:>15,.0f}")
    print(f"  ratio (1p/2p) = {Q_one/Q_two:.2f}")
    print()


print("Dense matrices:")
compare_costs(1000, 100, 1000, density=1.0)

print("Sparse matrices (10% density):")
compare_costs(1000, 100, 1000, density=0.1)

print("Large k:")
compare_costs(100, 10000, 100, density=1.0)